# 25/25Hz＋5fps：事务边界与轻量日志候选
保留原版。心率仍使用原PL＋Native100；新版4×3视频运动、MPU融合和混合显示不变。原IPC完整保留；复用已验证的轻量标量日志转换，原CSV字节、写入/flush及错误策略不变。
保护范围明确调整：原C事务40ms截止直到验证/ACK，Python返回沿用原整拍1秒；RTL与初始化/停止20ms、1秒新鲜度及退出、原队列与GC不变。
前两单元无硬件。最后单元由你手动一次60秒；先给屏幕独立供电并放稳手指。其它任务须结束；失败保留、不重试，收尾时不要中断内核。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
from IPython.display import display, FileLink
assert sys.platform == 'linux', '仅在板上运行'
root = Path.cwd()
assert root.name == 'v3b25 journal scalar60', '请在新候选目录运行'
from journal60_guard import verify
checked = verify(root)
print({k: checked[k] for k in ('bundle_validated','source_members_unchanged','package_members','manifest_sha256')})
from run_journal60 import short, prepare, run
short_report = short(expected_manifest_sha256=checked['manifest_sha256'])
if (root/'lean-ps-short-results.zip').is_file():
    display(FileLink('lean-ps-short-results.zip'))
assert short_report['passed'], '无硬件短检查未通过，保留结果，不启动硬件'
identity = short_report['ready_info']['journal60_ps']
assert identity['manifest_sha256'] == checked['manifest_sha256'] and identity['manifest_verified_before_original']
print('原700点READY秒数：', short_report['ready_info']['warmup_seconds'])
print('原PS自然退出检查通过；未加载电路、采集或视频。')


In [ ]:
# 只准备浏览器波形，不启动硬件。
prepared = prepare(expected_manifest_sha256=checked['manifest_sha256'])


In [ ]:
assert short_report['passed'], '先通过原无硬件短检查'
assert not globals().get('_journal60_used', False), '已尝试，保留结果，不重跑'
confirmation = input('其他任务已结束，屏幕已供电、手指放稳；仅运行一次60秒。输入 RUN-JOURNAL60: ')
assert confirmation == 'RUN-JOURNAL60', '未确认，不启动硬件'
_journal60_used = True
print('700点预热另计、正式60秒；原C40ms／整拍1秒，失败保存，不自动重试。', flush=True)
try:
    report = run(hardware=True, confirmation=confirmation, expected_manifest_sha256=checked['manifest_sha256'])
finally:
    if (root/'journal60-results.zip').is_file():
        display(FileLink('journal60-results.zip'))
    else:
        print('收尾/导出未确认，保留输出，不启动另一入口。')
